# Three-Class Next-Day AQI Model Comparison

This notebook compares the model families tested for the project’s three-class next-day AQI task using the current MySQL `air_quality.aqi_daily` table. It includes XGBoost, Random Forest, Extra Trees, HistGradientBoosting, Logistic Regression, CatBoost, LightGBM, and an XGBoost heavy-alert reviewer cascade.

The target groups are **Clean/good (AQI ≤ 100)**, **Light-to-moderate polluted (101–200)**, and **Heavily polluted (> 200)**. AQI labels are recomputed from all pollutant fields using the project’s shared AQI rules. Ordinary classifiers use today’s six pollutant summaries and nine weather summaries to predict the same station’s next calendar-day category.

The database is read-only. The notebook does not save a model, CSV, or database changes.

## Evaluation design

- Chronological split by unique dates: first 80% for training, final 20% for one held-out test period. All station rows for a date stay together.
- Tune parameters only within the training period using four expanding-window time-series folds and macro-F1.
- Compare accuracy, macro/weighted F1, balanced accuracy, and heavy-class precision, recall, F1, and false alerts.
- Include majority-class and persistence baselines. Persistence repeats today’s AQI class for tomorrow.
- The cascade runs a binary reviewer only when full-input XGBoost predicts the heavy class. The reviewer uses PM2.5 and weather; it either retains the alert or falls back to XGBoost’s most likely non-heavy class. Its threshold is chosen from training-period out-of-fold predictions.

Accuracy alone can hide missed heavy days. Review heavy-class recall and false alerts alongside overall scores.

## Why these model families were selected

The inputs are mostly numeric, tabular station-day measurements. Tree ensembles are practical candidates for nonlinear relationships and interactions in this kind of data: XGBoost is the project’s current boosted-tree reference; Random Forest and Extra Trees provide bagged-tree comparisons; HistGradientBoosting, CatBoost, and LightGBM provide other boosting implementations. Logistic Regression is included as a simpler linear baseline.

This is a focused comparison, not a test of every possible algorithm. SVMs and neural networks were not included because the first priority was comparing several strong tabular baselines with manageable tuning time. Trying more algorithms may change rankings, but it cannot create information absent from the inputs or automatically solve rare-class scarcity. All candidates should be judged on the same chronological holdout and on the class-level metrics that matter for the app.

For the three-class task, the main alert tradeoff is between detecting the rare heavily polluted class and limiting false alerts.

## Install CatBoost and LightGBM if needed

Run this in a notebook cell using the same kernel, then restart the kernel:

```python
%pip install catboost lightgbm
```

This notebook also requires SQLAlchemy/PyMySQL, scikit-learn, XGBoost, pandas, and NumPy.

## 1. Imports and project settings

The shared AQI helper is located from the repository tree. Set `AIR_QUALITY_DB_URL` (preferred) or `DB_URL` in the environment if your MySQL settings differ from the local default.

In [ ]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sqlalchemy import create_engine, text

from sklearn.base import clone
from sklearn.ensemble import (
    ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score, precision_recall_fscore_support,
)
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

try:
    from catboost import CatBoostClassifier
    from lightgbm import LGBMClassifier
except ImportError as error:
    raise ImportError(
        "Install the optional models in this notebook kernel with "
        "%pip install catboost lightgbm, then restart the kernel."
    ) from error

HERE = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in [HERE, *HERE.parents]
     if (p / "air-quality-classification" / "app" / "aqi_labels.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from the Data_Science repository or a subfolder.")
APP_DIR = PROJECT_ROOT / "air-quality-classification" / "app"
if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))
from aqi_labels import O3_1H_BREAKPOINTS, POLLUTANT_BREAKPOINTS, interpolate_iaqi

DB_URL = os.environ.get(
    "AIR_QUALITY_DB_URL",
    os.environ.get("DB_URL", "mysql+pymysql://root:@localhost/air_quality"),
)
SOURCE_TABLE = "aqi_daily"
POLLUTANTS = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3"]
WEATHER = [
    "TEMP_mean", "TEMP_range", "PRES_mean", "DEWP_mean", "RH_mean",
    "RAIN_sum", "WSPM_mean", "wd_sin", "wd_cos",
]
FEATURES = POLLUTANTS + WEATHER
REVIEW_FEATURES = ["PM2.5"] + WEATHER
CLASS_NAMES = ["Clean / good", "Light-to-moderate polluted", "Heavily polluted"]
RANDOM_STATE = 42
print("Project:", PROJECT_ROOT)
print("Table:", SOURCE_TABLE, "| full features:", len(FEATURES), "| reviewer:", len(REVIEW_FEATURES))

## 2. Read and validate `aqi_daily`

This cell runs a `SELECT` only, checks the fields required for features and labels, and verifies one row per station and date.

In [ ]:
required = ["date", "station", "station_id", *POLLUTANTS, "O3_1h_max", *WEATHER]
engine = create_engine(DB_URL, pool_pre_ping=True)
try:
    with engine.connect() as connection:
        source = pd.read_sql_query(text(f"SELECT * FROM {SOURCE_TABLE}"), connection)
finally:
    engine.dispose()

missing = [column for column in required if column not in source.columns]
if missing:
    raise KeyError(f"{SOURCE_TABLE} is missing required columns: {missing}")
source["date"] = pd.to_datetime(source["date"], errors="coerce").dt.normalize()
if source.duplicated(["station_id", "date"]).any():
    raise ValueError("aqi_daily contains duplicate station-date keys.")
for column in [*POLLUTANTS, "O3_1h_max", *WEATHER]:
    source[column] = pd.to_numeric(source[column], errors="coerce")

print(f"Rows: {len(source):,}; stations: {source.station.nunique():,}")
print(f"Date range: {source.date.min().date()} through {source.date.max().date()}")
display(source.head())

## 3. Rebuild AQI and exact next-day targets

AQI labels use all pollutants. The merge shifts the target date by exactly one calendar day for the same station, so missing dates are not treated as consecutive days.

In [ ]:
daily = source.copy()
iaqi_columns = []
for pollutant in POLLUTANTS:
    values = daily[pollutant]
    iaqi_column = f"{pollutant}_iaqi"
    if pollutant == "O3":
        ozone_8h = interpolate_iaqi(values, POLLUTANT_BREAKPOINTS["O3"])
        ozone_1h = interpolate_iaqi(daily["O3_1h_max"], O3_1H_BREAKPOINTS)
        use_one_hour = values.gt(800)
        ozone_8h.loc[use_one_hour] = ozone_1h.loc[use_one_hour]
        daily[iaqi_column] = ozone_8h.round()
    else:
        daily[iaqi_column] = interpolate_iaqi(
            values, POLLUTANT_BREAKPOINTS[pollutant]
        ).round()
    iaqi_columns.append(iaqi_column)

daily["AQI"] = daily[iaqi_columns].max(axis=1, skipna=True)
today = daily[["date", "station_id", "station", *FEATURES, "AQI"]].copy()
tomorrow = daily[["date", "station_id", "AQI"]].copy()
tomorrow["date"] = tomorrow["date"] - pd.Timedelta(days=1)
tomorrow = tomorrow.rename(columns={"AQI": "target_AQI"})
data = today.merge(
    tomorrow, on=["date", "station_id"], how="inner", validate="one_to_one"
).dropna(subset=["target_AQI"]).copy()
data["target_class"] = np.select(
    [data.target_AQI <= 100, data.target_AQI <= 200], [0, 1], default=2
).astype(int)
data["today_class"] = np.select(
    [data.AQI <= 100, data.AQI <= 200], [0, 1], default=2
).astype(float)
data.loc[data.AQI.isna(), "today_class"] = np.nan
data = data.sort_values(["date", "station"]).reset_index(drop=True)

print(f"Exact next-calendar-day examples: {len(data):,}")
display(data.target_class.map(dict(enumerate(CLASS_NAMES))).value_counts().rename("rows").to_frame())
display(data[["date", "station", "target_AQI", "target_class"]].head())

## 4. Chronological train/test split and date-grouped folds

The final 20% of unique dates are held out. All tuning uses only the first 80%.

In [ ]:
unique_dates = np.array(sorted(data.date.dropna().unique()))
cut = int(0.80 * len(unique_dates))
train_dates, test_dates = unique_dates[:cut], unique_dates[cut:]
train = data[data.date.isin(train_dates)].copy().reset_index(drop=True)
test = data[data.date.isin(test_dates)].copy().reset_index(drop=True)

train_date_positions = pd.factorize(train.date, sort=True)[0]
folds = []
for earlier, later in TimeSeriesSplit(n_splits=4).split(np.unique(train_date_positions)):
    folds.append((
        np.flatnonzero(np.isin(train_date_positions, earlier)),
        np.flatnonzero(np.isin(train_date_positions, later)),
    ))

print(f"Training: {len(train):,} rows, {train.date.min().date()} to {train.date.max().date()}")
print(f"Testing: {len(test):,} rows, {test.date.min().date()} to {test.date.max().date()}")
print(f"Four expanding-window folds over {len(train_dates):,} unique training dates.")

## 5. Define classifiers and search spaces

All searches use the same folds and macro-F1 scoring. The tree alternatives and logistic regression use class balancing; XGBoost matches the project’s unweighted multiclass setup. CatBoost disables model-file output.

In [ ]:
XGB_GRID = {
    "classifier__n_estimators": [120, 240],
    "classifier__max_depth": [3, 5],
    "classifier__learning_rate": [0.06],
}
TREE_GRID = {
    "classifier__n_estimators": [300, 600],
    "classifier__max_depth": [16, None],
    "classifier__min_samples_leaf": [3],
}
MODEL_SPECS = {
    "XGBoost": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("classifier", XGBClassifier(
                objective="multi:softprob", num_class=3, eval_metric="mlogloss",
                random_state=RANDOM_STATE, n_jobs=4, tree_method="hist", verbosity=0,
            )),
        ]), XGB_GRID,
    ),
    "Random Forest": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("classifier", RandomForestClassifier(
                random_state=RANDOM_STATE, n_jobs=-1,
                class_weight="balanced_subsample",
            )),
        ]), TREE_GRID,
    ),
    "Extra Trees": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("classifier", ExtraTreesClassifier(
                random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced",
            )),
        ]), TREE_GRID,
    ),
    "HistGradientBoosting": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("classifier", HistGradientBoostingClassifier(
                random_state=RANDOM_STATE, class_weight="balanced",
                early_stopping=False,
            )),
        ]), {
            "classifier__max_iter": [100, 200],
            "classifier__max_leaf_nodes": [15, 31],
            "classifier__learning_rate": [0.1],
        },
    ),
    "Logistic Regression": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
            ("classifier", LogisticRegression(
                max_iter=1500, class_weight="balanced", solver="lbfgs",
                random_state=RANDOM_STATE,
            )),
        ]), {"classifier__C": [0.1, 1, 10]},
    ),
    "CatBoost": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("classifier", CatBoostClassifier(
                loss_function="MultiClass", auto_class_weights="Balanced",
                random_seed=RANDOM_STATE, thread_count=4, verbose=False,
                allow_writing_files=False,
            )),
        ]), {
            "classifier__iterations": [300, 600],
            "classifier__depth": [4, 6],
            "classifier__learning_rate": [0.05],
        },
    ),
    "LightGBM": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("classifier", LGBMClassifier(
                objective="multiclass", class_weight="balanced",
                random_state=RANDOM_STATE, n_jobs=4, verbosity=-1,
                force_col_wise=True,
            )),
        ]), {
            "classifier__n_estimators": [200, 400],
            "classifier__num_leaves": [15, 31],
            "classifier__learning_rate": [0.05],
        },
    ),
}
print("Classifiers:", ", ".join(MODEL_SPECS))

## 6. Tune and evaluate each classifier

The held-out period is scored once after selecting each model’s parameters from training folds. Predictions use the model’s highest-probability class.

In [ ]:
def metric_row(name, y_true, prediction, cv_macro_f1=np.nan, best_params=None):
    p, r, f, support = precision_recall_fscore_support(
        y_true, prediction, labels=[2], zero_division=0
    )
    prediction = np.asarray(prediction).reshape(-1)
    y_true = np.asarray(y_true).reshape(-1)
    heavy_predictions = int(np.sum(prediction == 2))
    true_heavy_alerts = int(np.sum((prediction == 2) & (y_true == 2)))
    return {
        "model": name,
        "cv_macro_f1": cv_macro_f1,
        "accuracy": accuracy_score(y_true, prediction),
        "macro_f1": f1_score(y_true, prediction, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, prediction, average="weighted", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, prediction),
        "heavy_precision": p[0], "heavy_recall": r[0], "heavy_f1": f[0],
        "heavy_support": int(support[0]),
        "heavy_predictions": heavy_predictions,
        "heavy_true_positives": true_heavy_alerts,
        "heavy_false_alarms": heavy_predictions - true_heavy_alerts,
        "best_params": best_params or "-",
    }

y_test = test.target_class.to_numpy(dtype=int)
searches, fitted_models, predictions = {}, {}, {}
result_rows = []
for name, (pipeline, parameter_grid) in MODEL_SPECS.items():
    started = time.time()
    search = GridSearchCV(
        pipeline, parameter_grid, scoring="f1_macro", cv=folds,
        n_jobs=1, refit=True, return_train_score=False, verbose=0,
    )
    search.fit(train[FEATURES], train.target_class)
    best_model = search.best_estimator_
    probability = np.asarray(best_model.predict_proba(test[FEATURES]))
    classes = np.asarray(best_model.named_steps["classifier"].classes_, dtype=int)
    prediction = classes[probability.argmax(axis=1)].reshape(-1)
    searches[name], fitted_models[name], predictions[name] = search, best_model, prediction
    result_rows.append(metric_row(name, y_test, prediction, search.best_score_, search.best_params_))
    print(
        f"{name}: CV macro-F1={search.best_score_:.4f}; "
        f"test accuracy={accuracy_score(y_test, prediction):.4f}; "
        f"elapsed={time.time() - started:.1f}s; params={search.best_params_}"
    )

## 7. Add majority and persistence baselines

The majority baseline always predicts the most common training class. Persistence repeats today’s class; if today’s AQI cannot be calculated, the training majority is used for that row.

In [ ]:
majority_class = int(train.target_class.value_counts().idxmax())
majority_prediction = np.full(len(test), majority_class, dtype=int)
persistence_prediction = test.today_class.fillna(majority_class).to_numpy(dtype=int)
predictions["Majority baseline"] = majority_prediction
predictions["Persistence baseline"] = persistence_prediction
result_rows.extend([
    metric_row("Majority baseline", y_test, majority_prediction),
    metric_row("Persistence baseline", y_test, persistence_prediction),
])
results = pd.DataFrame(result_rows)
RESULT_COLUMNS = [
    "model", "cv_macro_f1", "accuracy", "macro_f1", "weighted_f1",
    "balanced_accuracy", "heavy_precision", "heavy_recall", "heavy_f1",
    "heavy_support", "heavy_predictions", "heavy_true_positives", "heavy_false_alarms",
]
display(results[RESULT_COLUMNS].sort_values("macro_f1", ascending=False).style.format({
    "cv_macro_f1": "{:.4f}", "accuracy": "{:.2%}", "macro_f1": "{:.4f}",
    "weighted_f1": "{:.4f}", "balanced_accuracy": "{:.2%}",
    "heavy_precision": "{:.2%}", "heavy_recall": "{:.2%}", "heavy_f1": "{:.4f}",
}, na_rep="—"))

## 8. XGBoost with a PM2.5-and-weather heavy-alert reviewer

The main XGBoost uses all 15 features. Only a class-2 prediction is sent to a binary reviewer using PM2.5 and weather. A rejected alert falls back to the main model’s most likely non-heavy class. The reviewer is class-weighted, and its threshold is selected using training-period out-of-fold predictions; the final test dates remain untouched.

In [ ]:
def class_probabilities(model, X, n_classes):
    raw = np.asarray(model.predict_proba(X))
    classes = np.asarray(model.named_steps["classifier"].classes_, dtype=int)
    aligned = np.zeros((len(X), n_classes), dtype=float)
    aligned[:, classes] = raw
    return aligned


def reviewer_f1_at_half(estimator, X, y):
    classes = np.asarray(estimator.named_steps["classifier"].classes_, dtype=int)
    positive_column = int(np.flatnonzero(classes == 1)[0])
    scores = estimator.predict_proba(X)[:, positive_column]
    return f1_score(y, (scores >= 0.5).astype(int), zero_division=0)

reviewer_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", XGBClassifier(
        objective="binary:logistic", eval_metric="logloss",
        random_state=RANDOM_STATE, n_jobs=4, tree_method="hist", verbosity=0,
    )),
])
reviewer_search = GridSearchCV(
    reviewer_pipeline, XGB_GRID, scoring=reviewer_f1_at_half,
    cv=folds, n_jobs=1, refit=True, return_train_score=False,
)
reviewer_y = (train.target_class.to_numpy(dtype=int) == 2).astype(int)
reviewer_weights = compute_sample_weight("balanced", reviewer_y)
reviewer_search.fit(train[REVIEW_FEATURES], reviewer_y, classifier__sample_weight=reviewer_weights)
reviewer_model = reviewer_search.best_estimator_

oof_main = np.full((len(train), 3), np.nan)
oof_reviewer = np.full(len(train), np.nan)
for train_rows, validation_rows in folds:
    fold_train = train.iloc[train_rows]
    fold_validation = train.iloc[validation_rows]
    if fold_train.target_class.nunique() != 3:
        continue
    main_fold = clone(fitted_models["XGBoost"])
    main_fold.fit(fold_train[FEATURES], fold_train.target_class)
    oof_main[validation_rows] = class_probabilities(main_fold, fold_validation[FEATURES], 3)

    fold_y = (fold_train.target_class.to_numpy(dtype=int) == 2).astype(int)
    if np.unique(fold_y).size < 2:
        continue
    reviewer_fold = clone(reviewer_model)
    weights = compute_sample_weight("balanced", fold_y)
    reviewer_fold.fit(fold_train[REVIEW_FEATURES], fold_y, classifier__sample_weight=weights)
    classes = np.asarray(reviewer_fold.named_steps["classifier"].classes_, dtype=int)
    positive_column = int(np.flatnonzero(classes == 1)[0])
    oof_reviewer[validation_rows] = reviewer_fold.predict_proba(
        fold_validation[REVIEW_FEATURES]
    )[:, positive_column]

valid = np.isfinite(oof_reviewer) & np.isfinite(oof_main).all(axis=1)
oof_y = train.target_class.to_numpy(dtype=int)[valid]
oof_primary = oof_main[valid].argmax(axis=1)
oof_scores = oof_reviewer[valid]
threshold_results = []
for threshold in np.linspace(0.05, 0.95, 91):
    routed = oof_primary.copy()
    rejected = (oof_primary == 2) & (oof_scores < threshold)
    if rejected.any():
        routed[rejected] = oof_main[valid][rejected, :2].argmax(axis=1)
    p, r, f, _ = precision_recall_fscore_support(oof_y, routed, labels=[2], zero_division=0)
    threshold_results.append((float(f[0]), float(p[0]), float(threshold), float(r[0])))
threshold_f1, threshold_precision, reviewer_threshold, threshold_recall = max(
    threshold_results, key=lambda row: (row[0], row[1], row[2])
)
print("Reviewer best CV F1 at threshold .50:", round(reviewer_search.best_score_, 4))
print("Reviewer parameters:", reviewer_search.best_params_)
print("Training OOF threshold:", round(reviewer_threshold, 3))
print("Training OOF routed heavy precision/recall/F1:",
      round(threshold_precision, 4), round(threshold_recall, 4), round(threshold_f1, 4))

In [ ]:
test_main_probability = class_probabilities(fitted_models["XGBoost"], test[FEATURES], 3)
cascade_prediction = test_main_probability.argmax(axis=1)
reviewer_classes = np.asarray(reviewer_model.named_steps["classifier"].classes_, dtype=int)
positive_column = int(np.flatnonzero(reviewer_classes == 1)[0])
test_reviewer_scores = reviewer_model.predict_proba(test[REVIEW_FEATURES])[:, positive_column]
main_heavy_alerts = cascade_prediction == 2
rejected_alerts = main_heavy_alerts & (test_reviewer_scores < reviewer_threshold)
if rejected_alerts.any():
    cascade_prediction[rejected_alerts] = test_main_probability[rejected_alerts, :2].argmax(axis=1)
predictions["XGBoost + heavy reviewer"] = cascade_prediction
cascade_row = metric_row(
    "XGBoost + heavy reviewer", y_test, cascade_prediction,
    best_params={"reviewer_threshold": round(float(reviewer_threshold), 4)},
)
results = pd.concat([results, pd.DataFrame([cascade_row])], ignore_index=True)
print("Main heavy alerts:", int(main_heavy_alerts.sum()))
print("Reviewer accepted:", int((main_heavy_alerts & ~rejected_alerts).sum()))
print("Reviewer rejected:", int(rejected_alerts.sum()))
display(results[RESULT_COLUMNS].sort_values("macro_f1", ascending=False))

## 9. Per-class reports and confusion matrices

Rows in each confusion matrix are actual classes; columns are predicted classes.

In [ ]:
for name, prediction in predictions.items():
    print(f"\n{name}")
    report = classification_report(
        y_test, prediction, labels=[0, 1, 2], target_names=CLASS_NAMES,
        output_dict=True, zero_division=0,
    )
    display(pd.DataFrame(report).T.round(4))
    display(pd.DataFrame(
        confusion_matrix(y_test, prediction, labels=[0, 1, 2]),
        index=[f"Actual: {label}" for label in CLASS_NAMES],
        columns=[f"Predicted: {label}" for label in CLASS_NAMES],
    ))

## 10. Final recommendation

The next cell ranks the measured classifiers by accuracy, macro-F1, heavy-class F1, and recall. Use the metric that matches the app’s purpose rather than accuracy alone.

In [ ]:
model_results = results[~results.model.isin(["Majority baseline", "Persistence baseline"])].copy()
best_accuracy = model_results.loc[model_results.accuracy.idxmax()]
best_macro = model_results.loc[model_results.macro_f1.idxmax()]
best_heavy_f1 = model_results.loc[model_results.heavy_f1.idxmax()]
best_heavy_recall = model_results.loc[model_results.heavy_recall.idxmax()]
print(f"Best accuracy: {best_accuracy.model} ({best_accuracy.accuracy:.2%})")
print(f"Best macro-F1: {best_macro.model} ({best_macro.macro_f1:.4f})")
print(
    f"Best heavy-class F1: {best_heavy_f1.model} ({best_heavy_f1.heavy_f1:.4f}); "
    f"{int(best_heavy_f1.heavy_true_positives)} correct alerts and "
    f"{int(best_heavy_f1.heavy_false_alarms)} false alerts."
)
print(
    f"Highest heavy recall: {best_heavy_recall.model} "
    f"({best_heavy_recall.heavy_recall:.2%}) with "
    f"{int(best_heavy_recall.heavy_false_alarms)} false alerts."
)
print("\nRecommendation depends on the alert tradeoff:")
print("- Use the highest macro-F1/balanced-accuracy model for more even performance across classes.")
print("- Use higher heavy recall only if the resulting false-alert volume is acceptable.")
print("- Prefer fewer false alerts when alert credibility is the main concern.")
print("- Treat the chronological holdout as historical evidence, not a guarantee for future dates.")

## Results snapshot from the latest in-memory benchmark

Using the current 2013–2025 `aqi_daily` table and the September 2022–January 3, 2025 holdout, **Extra Trees** had the strongest balance among the tested classifiers: macro-F1 **0.550**, balanced accuracy **55.80%**, and heavy-class F1 **0.179**. It caught 28 of 141 heavy days but produced 144 false heavy alerts.

**XGBoost** had the highest overall accuracy (**78.27%**) and fewer false heavy alerts (**40**), but caught only 9 of 141 heavy days. This makes XGBoost preferable when limiting false alerts and Extra Trees preferable when the goal is better class-balanced performance and more heavy-day detection. Neither is a universal winner; the decision depends on the acceptable false-alert rate.

CatBoost and LightGBM caught more heavy days than XGBoost, but generated more false alerts and did not lead on accuracy or macro-F1. Rerun this notebook to refresh results from the current database; the dynamic recommendation above will use the newly measured scores.